# RNN/LSTM Decoder Training — Flickr8k

Pre-inject architecture (Vinyals et al., 2015).  
Trains **12 Keras models**: 6 SimpleRNN × 6 LSTM (layers ∈ {1,2,3} × hidden ∈ {128,512}).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import json
import time
from pathlib import Path
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from src.captioning.preprocessing import (
    run_preprocessing, load_vocab, load_split_ids, build_sequences
)

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

C:\Users\saman\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.4.3)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0
GPU: []


## 1 — Preprocessing

In [ ]:
VOCAB_FILE = Path('../data/flickr8k/vocab.json')
MAX_LEN    = 35   # includes <start> and <end>

if VOCAB_FILE.exists():
    print('Start Loading vocab and splits from disk...')
    word2idx, idx2word = load_vocab(VOCAB_FILE)
    from src.captioning.preprocessing import load_captions, load_split_ids
    captions = load_captions('../data/flickr8k/captions.txt')
    train_ids, val_ids, test_ids = load_split_ids('../data/flickr8k')
else:
    import os; os.chdir('..')
    captions, word2idx, idx2word, train_ids, val_ids, test_ids = run_preprocessing(max_len=MAX_LEN)
    os.chdir('notebooks')

VOCAB_SIZE = len(word2idx)
print(f'Vocab size: {VOCAB_SIZE}')
print(f'Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}')

Loading captions...
  8092 images, 40456 captions total
  Split: 6000 train / 1000 val / 1092 test
Vocab saved to data\flickr8k\vocab.json  (7449 tokens)
  Vocabulary size: 7449
Vocab size: 7449
Train: 6000 | Val: 1000 | Test: 1092


In [3]:
# Load CNN features
FEATURES_FILE = Path('../data/features/flickr8k_features.npy')
features = np.load(FEATURES_FILE, allow_pickle=True).item()  # dict {fname: (2048,)}
print(f'Features loaded: {len(features)} images')

Features loaded: 8091 images


## 2 — Data Generator

In [ ]:
from src.captioning.preprocessing import build_sequences, clean_caption, tokenize_caption

def make_dataset(img_ids, sequences, features, word2idx, max_len, batch_size=64, shuffle=True):
    """
    Build a tf.data.Dataset that yields (inputs, targets) per batch.
    """
    img_feats = np.array([features[fname] for fname in img_ids], dtype=np.float32)
    cap_in    = sequences[:, :-1]    # (N, max_len-1)
    cap_out   = sequences[:, 1:]     # (N, max_len-1)

    ds = tf.data.Dataset.from_tensor_slices(((img_feats, cap_in), cap_out))
    if shuffle:
        ds = ds.shuffle(buffer_size=10000, seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


print('Start building train sequences')
train_img_ids, train_seqs = build_sequences(captions, train_ids, word2idx, MAX_LEN)
print('Start building val sequences')
val_img_ids,   val_seqs   = build_sequences(captions, val_ids,   word2idx, MAX_LEN)

print(f'Train pairs: {len(train_seqs)} | Val pairs: {len(val_seqs)}')

Building train sequences...
Building val sequences...
Train pairs: 29996 | Val pairs: 5000


## 3 — Model Builder

In [ ]:
def build_decoder(decoder_type, num_layers, hidden_units, vocab_size, embed_dim=256, max_len=35):
    """
    Pre-inject decoder using Keras Functional API.

    decoder_type : 'rnn' | 'lstm'
    num_layers   : 1, 2, or 3
    hidden_units : 128 or 512
    """
    seq_len = max_len - 1

    #  Inputs 
    img_input = keras.Input(shape=(2048,), name='img_feat')
    cap_input = keras.Input(shape=(seq_len,), dtype='int32', name='caption_in')

    #  CNN feature projection
    img_proj = layers.Dense(embed_dim, activation='linear', name='dense_proj')(img_input)
    img_proj = layers.Reshape((1, embed_dim))(img_proj)   # (batch, 1, embed_dim)

    #  Caption embedding 
    cap_emb = layers.Embedding(vocab_size, embed_dim, name='embedding')(cap_input)
                                                            # (batch, seq_len, embed_dim)

    #  Concatenate: x_{-1} prepended to caption embeddings 
    x = layers.Concatenate(axis=1)([img_proj, cap_emb])   # (batch, seq_len+1, embed_dim)

    #  Recurrent layers 
    RNNCell = layers.SimpleRNN if decoder_type == 'rnn' else layers.LSTM
    rnn_name = 'simple_rnn' if decoder_type == 'rnn' else 'lstm'

    for i in range(num_layers):
        name   = f'{rnn_name}' if i == 0 else f'{rnn_name}_{i}'
        ret_seq = (i < num_layers - 1)   # all intermediate layers return sequences
        x = RNNCell(hidden_units, return_sequences=True, name=name)(x)
    # last layer also returns sequences so Dense applies per-timestep

    #  Output Dense: per timestep → vocab_size 
    output = layers.Dense(vocab_size, activation='softmax', name='dense_out')(x)
                                                            # (batch, seq_len+1, vocab_size)
    # Slice off the x_{-1} output (we predict starting from t=0)
    output = output[:, 1:, :]   # (batch, seq_len, vocab_size)

    model = keras.Model(inputs=[img_input, cap_input], outputs=output)
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick sanity check
test_m = build_decoder('lstm', 1, 128, VOCAB_SIZE, max_len=MAX_LEN)
test_m.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_feat            │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_proj (Dense)  │ (None, 256)       │    524,544 │ img_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_in          │ (None, 34)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 256)    │          0 │ dense_proj[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 34, 256)   │  1,906,944 │ caption_in[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 35, 256)   │          0 │ reshape[0][0],    │
│ (Concatenate)       │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 35, 128)   │    197,120 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_out (Dense)   │ (None, 35, 7449)  │    960,921 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 34, 7449)  │          0 │ dense_out[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,589,529 (13.69 MB)

 Trainable params: 3,589,529 (13.69 MB)

 Non-trainable params: 0 (0.00 B)

## 4 — Training Loop (all 12 variants)

In [6]:
EMBED_DIM  = 256
EPOCHS     = 20
BATCH_SIZE = 64
MODEL_DIR  = Path('../models/rnn')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    # (decoder_type, num_layers, hidden_units)
    ('rnn',  1, 128), ('rnn',  1, 512),
    ('rnn',  2, 128), ('rnn',  2, 512),
    ('rnn',  3, 128), ('rnn',  3, 512),
    ('lstm', 1, 128), ('lstm', 1, 512),
    ('lstm', 2, 128), ('lstm', 2, 512),
    ('lstm', 3, 128), ('lstm', 3, 512),
]

history_all = {}   # config_name → history dict

train_ds = make_dataset(train_img_ids, train_seqs, features, word2idx,
                        MAX_LEN, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_img_ids,   val_seqs,   features, word2idx,
                        MAX_LEN, BATCH_SIZE, shuffle=False)

for dec_type, n_layers, n_hidden in CONFIGS:
    cfg_name  = f'{dec_type}_L{n_layers}_H{n_hidden}'
    save_path = MODEL_DIR / f'{cfg_name}.keras'

    if save_path.exists():
        print(f'[SKIP] {cfg_name} already trained.')
        continue

    print(f'\n=== Training {cfg_name} ===')
    model = build_decoder(dec_type, n_layers, n_hidden, VOCAB_SIZE,
                          embed_dim=EMBED_DIM, max_len=MAX_LEN)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ModelCheckpoint(str(save_path), monitor='val_loss', save_best_only=True),
    ]

    t0  = time.time()
    hist = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=callbacks, verbose=1)
    elapsed = time.time() - t0

    history_all[cfg_name] = {
        'loss':     hist.history['loss'],
        'val_loss': hist.history['val_loss'],
        'train_time_s': elapsed,
    }
    print(f'  Saved → {save_path}  ({elapsed:.0f}s)')

# Save all histories
with open(MODEL_DIR / 'training_histories.json', 'w') as f:
    json.dump(history_all, f, indent=2)
print('\nAll training complete.')

KeyError: 'image'

## 5 — Loss Curves

In [ ]:
# Load histories if re-running
hist_file = MODEL_DIR / 'training_histories.json'
if hist_file.exists():
    with open(hist_file) as f:
        history_all = json.load(f)

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
axes = axes.flatten()

for ax, (cfg, h) in zip(axes, history_all.items()):
    ax.plot(h['loss'],     label='train')
    ax.plot(h['val_loss'], label='val')
    ax.set_title(cfg, fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training / Validation Loss — All Configs', fontsize=13)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'loss_curves_all.png', dpi=120)
plt.show()

In [ ]:
# Summary table: best val_loss and training time
print(f"{'Config':<22} {'Best val_loss':>14} {'Epochs':>7} {'Train time':>12}")
print('-' * 58)
for cfg, h in sorted(history_all.items(), key=lambda x: min(x[1]['val_loss'])):
    best   = min(h['val_loss'])
    epochs = len(h['val_loss'])
    mins   = h.get('train_time_s', 0) / 60
    print(f"{cfg:<22} {best:>14.4f} {epochs:>7} {mins:>10.1f}m")